## Objetivo do notebook
#### O objetivo deste notebook é realizar a leitura de dados das origens do dataset notas na camada bronze e gravar de forma versionada na camada silver


### Import Libs

In [1]:
!pip install minio

In [2]:
from pyspark.sql import SparkSession
from datetime import datetime
from minio import Minio
from pyspark.sql.functions import to_date, when, col, lit,isnan
from io import BytesIO

import os
import pandas as pd
import glob
import pyarrow.parquet as pq

### Definição de variáveis

In [3]:
# Sessão Spark
spark = SparkSession.builder.getOrCreate()

# Parametros de input e output das origens
camadaLeitura = 'bronze'
camadaEscrita = 'silver'
pasta = 'notas'
temp_blobs = '/home/jovyan/notebooks/_temporary_blobs/' # pasta temporária para armazenamento de objetos

# Conexão ao miniIO
minio_endpoint = 'minio:9000'
minio_access_key = 'minioaccesskey'
minio_secret_key = 'miniosecretkey'
minio_client = Minio(minio_endpoint, access_key=minio_access_key, secret_key=minio_secret_key, secure=False)

# Definição de variáveis para versionamento dos dados na camada bronze
timestamp = datetime.today().strftime('%Y%m%d')
minio_path = f'{pasta}/{pasta + timestamp}'

### Leitura da camada Bronze

In [4]:
dataframes = []

try:
    # Lista objetos no bucket
    objects = minio_client.list_objects(camadaLeitura, prefix=pasta, recursive=True)

    # Itera sobre os objetos na pasta
    for obj in objects:
        # Lê o conteúdo do objeto
        content = minio_client.get_object(camadaLeitura, obj.object_name).read()
        
        # Lê o conteúdo do objeto Parquet usando pyarrow e converte para DataFrame Pandas
        df_pandas = pq.read_table(BytesIO(content)).to_pandas()
        
        # Adiciona o DataFrame Pandas à lista de DataFrames
        dataframes.append(df_pandas)
        
        print(f"Objeto encontrado: {obj.object_name}")
except error.MinioException as e:
    print(f"Erro ao listar objetos: {e}")

# Concatena todos os DataFrames Pandas em um único DataFrame Pandas
consolidated_df_pandas = pd.concat(dataframes, ignore_index=True)

# Converte para DataFrame PySpark
df_matri_notas = spark.createDataFrame(consolidated_df_pandas)

Objeto encontrado: notas / notas20240114/part-00000-07cca1db-1bf1-48fa-9b11-062a57eb1d56-c000.snappy.parquet
Objeto encontrado: notas / notas20240114/part-00000-6bddb577-5319-47b0-b038-1bb3d6c9e4a3-c000.snappy.parquet
Objeto encontrado: notas / notas20240114/part-00000-8112c715-009d-429b-8ac4-a1eb61840680-c000.snappy.parquet
Objeto encontrado: notas / notas20240114/part-00001-07cca1db-1bf1-48fa-9b11-062a57eb1d56-c000.snappy.parquet
Objeto encontrado: notas / notas20240114/part-00001-6bddb577-5319-47b0-b038-1bb3d6c9e4a3-c000.snappy.parquet
Objeto encontrado: notas / notas20240114/part-00001-8112c715-009d-429b-8ac4-a1eb61840680-c000.snappy.parquet
Objeto encontrado: notas / notas20240114/part-00002-07cca1db-1bf1-48fa-9b11-062a57eb1d56-c000.snappy.parquet
Objeto encontrado: notas / notas20240114/part-00002-6bddb577-5319-47b0-b038-1bb3d6c9e4a3-c000.snappy.parquet
Objeto encontrado: notas / notas20240114/part-00002-8112c715-009d-429b-8ac4-a1eb61840680-c000.snappy.parquet
Objeto encontrado: 

### Tratamento dos dados

In [5]:
df_matri_notas = df_matri_notas.select(
    'id_matricula',
    'id_disciplina',
    # Tipagem da coluna data
    to_date('data').alias('data'),
    # Retorna zero caso não tenha nota
    when(isnan(col('nota')), lit(0)).otherwise(col('nota')).alias('nota'),
    'etl_date'
)

### Gravação do dataframe em um diretório temporário

In [6]:
df_matri_notas.write.parquet(temp_blobs, mode="overwrite")

### Gravação na camada Silver

In [7]:
# Recria bucket caso não exista
if not minio_client.bucket_exists(camadaEscrita):
    minio_client.make_bucket(camadaEscrita)

# Lista todos os arquivos Parquet no diretório temporário de blobs
arquivos_parquet = glob.glob(os.path.join(temp_blobs, '*.parquet'))

# Itera sobre a lista de arquivos Parquet e envia cada um para o MinIO
for arquivo_parquet in arquivos_parquet:
    try:
        # Envia o arquivo para a camada bronze
        nome_arquivo = os.path.basename(arquivo_parquet)
        minio_client.fput_object(camadaEscrita, os.path.join(minio_path, nome_arquivo), arquivo_parquet)
        print(f"Arquivo '{nome_arquivo}' enviado com sucesso para o MinIO em '{os.path.join(minio_path, nome_arquivo)}'.")
    except S3Error as e:
        print(f"Erro ao enviar o arquivo para o MinIO: {e}")

Arquivo 'part-00000-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet' enviado com sucesso para o MinIO em 'notas/notas20240114/part-00000-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet'.
Arquivo 'part-00001-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet' enviado com sucesso para o MinIO em 'notas/notas20240114/part-00001-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet'.
Arquivo 'part-00002-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet' enviado com sucesso para o MinIO em 'notas/notas20240114/part-00002-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet'.
Arquivo 'part-00003-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet' enviado com sucesso para o MinIO em 'notas/notas20240114/part-00003-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet'.
Arquivo 'part-00004-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet' enviado com sucesso para o MinIO em 'notas/notas20240114/part-00004-8b380bf8-bbd8-4084-81d1-b8edfda17c

### Remove dados do diretório temporário de blobs

In [8]:
# Liste todos os arquivos na pasta
arquivos_na_pasta = os.listdir(temp_blobs)

# Itere sobre os arquivos e os delete
for arquivo in arquivos_na_pasta:
    caminho_completo = os.path.join(temp_blobs, arquivo)
    try:
        if os.path.isfile(caminho_completo):
            os.remove(caminho_completo)
            print(f'{caminho_completo} deletado com sucesso.')
    except Exception as e:
        print(f'Erro ao deletar {caminho_completo}: {e}')

spark.stop()

/home/jovyan/notebooks/_temporary_blobs/.part-00000-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet.crc deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/.part-00001-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet.crc deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/.part-00002-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet.crc deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/.part-00003-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet.crc deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/.part-00004-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet.crc deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/.part-00005-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet.crc deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/.part-00006-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet.crc deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/.part-00007-8b3

/home/jovyan/notebooks/_temporary_blobs/part-00000-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/part-00001-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/part-00002-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/part-00003-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/part-00004-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/part-00005-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/part-00006-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c000.snappy.parquet deletado com sucesso.
/home/jovyan/notebooks/_temporary_blobs/part-00007-8b380bf8-bbd8-4084-81d1-b8edfda17c01-c0